# Figure 18 -- convergence and degeneracy

Loads `bench/results/density_reconstruction/reconstruction_runs.json`, produced by `bench/payoff_static/reconstruction_runs.py`. No computation here.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == "jaccpot_paper" else pathlib.Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt

from examples.jaccpot_paper.common import jsonio, style

style.apply()
FIG_DIR = jsonio.RESULTS_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
art = jsonio.read_result("density_reconstruction/reconstruction_runs.json")
cfg, recs = art["config"], art["data"]["records"]
recs = [r for r in recs if not r.get("failed")]
if not recs:
    raise SystemExit("reconstruction_runs.json has no successful rows")

fig, axes = style.figure(width=style.TWO_COL, height=2.9, ncols=2)

# -- left: loss against iteration, one curve per initial guess ------------- #
ax = axes[0]
guesses = [g for g in cfg["initial_guesses"]]
for i, guess in enumerate(guesses):
    sel = [r for r in recs if r["parameterization"] == "positions"
           and r["initial_guess"] == guess and r["regularized"]]
    if not sel:
        continue
    run = sel[0]
    trace = run["loss_trace"]
    ax.plot([t["iteration"] for t in trace], [t["loss"] for t in trace],
            color=style.CATEGORICAL[i % len(style.CATEGORICAL)],
            label=guess.replace("_", " "))
ax.set_yscale("log")
ax.set_xlabel("iteration"); ax.set_ylabel("loss (field residual + penalties)")
style.finish(ax, legend=True, legend_kwargs={"loc": "upper right", "fontsize": 5.6})

# -- right: the divergence -- field improves, density saturates ------------ #
# This is fig18's content. Plotted as the RATIO of each metric to its own
# starting value, so two quantities in different units share an axis honestly.
ax = axes[1]
for i, kind in enumerate(("parametric", "positions")):
    sel = sorted((r for r in recs if r["parameterization"] == kind and r["regularized"]),
                 key=lambda r: r["num_free_parameters"])
    if not sel:
        continue
    field_gain = [r["field_residual_after"]["rel_l2"] / r["field_residual_before"]["rel_l2"]
                  for r in sel]
    dens_gain = [r["density_after"]["grid_rel_l2"] / r["density_before"]["grid_rel_l2"]
                 for r in sel]
    ax.scatter(field_gain, dens_gain, marker=style.MARKERS[i], s=24,
               color=style.CATEGORICAL[i % len(style.CATEGORICAL)], label=kind)
ax.plot([1e-3, 2.0], [1e-3, 2.0], color=style.GRID, lw=0.8, zorder=0)
ax.axhline(1.0, color=style.GRID, lw=0.6, zorder=0)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("field residual, after / before")
ax.set_ylabel("density disagreement, after / before")
style.finish(ax, legend=True, legend_kwargs={"loc": "lower right", "fontsize": 5.6})

fig.tight_layout()
style.footer(fig, "%s, %d runs, N in %s, %d iterations" % (
    art["meta"]["device_kind"], len(recs), cfg["n"], cfg["iterations"]))
style.save(fig, str(FIG_DIR / "fig18_convergence_and_degeneracy.pdf"))


## Caption


Convergence, and the sensitivity to where the fit starts. **Left:** loss against
iteration for the high-dimensional fit from each of the four initial guesses --
a small perturbation of truth, an isotropised truth, a structurally wrong smooth
analytic model, and a naive uniform sphere. The loss is non-convex in the
positions, so the spread between these curves is a result in its own right and
not a nuisance. **Right:** the divergence that states the degeneracy
quantitatively. Each point is one run, plotting how much the field residual
improved against how much the density disagreement improved, each relative to
its own starting value. Points to the right of the diagonal improved the field
by more than they improved the density; points near the horizontal line at one
did not recover the density at all despite fitting the field. The
high-dimensional arm has vastly more freedom to move mass around at fixed field
than the parametric arm, and this is where that shows.
